<a href="https://colab.research.google.com/github/Farhad-Norouzzadeh/NanoGPT/blob/main/NanoGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Tiny GPT: From Text to a Language Model

In this notebook, we will build a very small language model from scratch using PyTorch.

The goal is to understand the basic ideas behind GPT-style models by starting with the simplest possible language model.

We will work with the **Tiny Shakespeare** dataset.

Our overall pipeline is:

1. Load the text dataset
2. Inspect the text
3. Build a character-level vocabulary
4. Convert characters into integers
5. Convert the entire dataset into a PyTorch tensor
6. Split the data into training and validation sets
7. Create input/target training examples
8. Create batches
9. Build a simple **Bigram Language Model**
10. Calculate predictions and the training loss

The important idea throughout the notebook is:

> Given some context, predict the next character.



## 1. Load and inspect the dataset

First, we load the Tiny Shakespeare text file.

A language model needs a large amount of text from which it can learn patterns.

At this stage, we are not doing any machine learning yet. We are simply loading the raw training corpus into a Python string.

In [ ]:
with open('/content/tiny-shakespeare.txt', 'r', encoding='utf-8') as f:
  text = f.read()

### Inspect the beginning of the dataset

looking at the first 100 characters of the dataset.

This allows us to verify that the text was loaded correctly and gives us an idea of what our model will be learning from.

In [ ]:
print(f"length of the characters {len(text)}")

length of the characters 1115394


## 2. Build the vocabulary

Our model cannot directly work with characters such as `"a"`, `"b"`, `" "` or `"!"`.

Neural networks work with numerical values.

First, we find all the **unique characters** that occur in the dataset.

We call this collection the **vocabulary**.

Because this is a **character-level language model**, each character is considered a token.

For example, our vocabulary might contain:

```text
a b c d ... z
A B C ... Z
space
newline
punctuation
...

In [ ]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [ ]:
# extract the unique characters from the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(chars) # this is the list of unique characters that we have in out dataset
print(''.join(chars))
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65




## 3. Create a character-to-integer mapping

We now need a way to convert characters into integers.

We create two dictionaries:

### `stoi`

```text
string → integer
```
### `itos`

```text
string → integer

In [ ]:
# create the index of each character
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
print(stoi)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}



## 4. Encode and decode text

Now we define two functions:

### `encode`

Converts a string into a list of integer token IDs.

For example:

```text
"hi"
might become: [46, 47]
```
### `decode`

Converts an integer token IDs into a string.

For example:

```text
[46, 47]
might become: "hi"
```


In [ ]:
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[c] for c in l)
print(encode("hi there"))
print(decode(encode("hi there")))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there



## 5. Convert the `entire` dataset into a PyTorch tensor

We now encode the complete Shakespeare dataset and store it as a PyTorch tensor.

```python
data = torch.tensor(encode(text), dtype=torch.long)
```
Each character is now represented by an integer.
for instance: <br>
```text
text:

"First"

↓ encode

[18, 47, 56, 57, 58]
```

In [ ]:
# Stroring the dataset into a tensor
import torch
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape)
print(data.type)
print(data[:100])

torch.Size([1115394])
<built-in method type of Tensor object at 0x7deb1dd09220>
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])



## 6. Split the dataset into training and validation data

We don't want to train and evaluate the model on exactly the same data.

So we split the dataset into two parts:

```text
90% → training data
10% → validation data

In [ ]:
# now lets split the data into training and validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

## 7. Define the context length

We now introduce:

```python
block_size = 8
```

the `block_size` represents the maximum number of previous characters that we use as context for a training example. <br>

for example, with: <br>
`block_size = 8` <br>
we can have a sequence such as:<br>
`F i r s t  c i` <br>
<br>
the model will learn the next character at every position.<br>



In [ ]:
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

# 8. Understanding Input and Target Sequences

We now arrive at one of the most important ideas in the entire notebook: **how do we turn raw text into a supervised learning problem?**

Our goal is to train a language model to predict the **next character**.

Remember that our text has already been converted into integer token IDs.

For example:

```text
"First"
```

might be represented as:

```text
[18, 47, 56, 57, 58]
```

We now take a small section of the training data:

```python
block_size = 8
```

This means that we will work with sequences containing up to 8 characters.

---

## Input (`x`) and Target (`y`)

We create:

```python
x = train_data[:block_size]
y = train_data[1:block_size+1]
```

The important thing to notice is that `y` is shifted by **one position** relative to `x`.

Conceptually:

```text
x:  [a, b, c, d, e, f, g, h]
y:  [b, c, d, e, f, g, h, i]
```

The input contains the characters that the model sees.

The target contains the character that the model should predict next.

Therefore:

```text
input → target

a       → b
b       → c
c       → d
d       → e
...
```

But because we want the model to use all the available context, we can think about the training examples as:

```text
context                     target

[a]                         b
[a, b]                      c
[a, b, c]                   d
[a, b, c, d]                e
[a, b, c, d, e]             f
[a, b, c, d, e, f]          g
[a, b, c, d, e, f, g]       h
[a, b, c, d, e, f, g, h]    i
```

This is exactly what the loop in the next part of the cell demonstrates.

---

## Why do we do this?

A neural network needs a clearly defined learning problem.

We cannot simply give it Shakespeare and say:

> "Learn language."

Instead, we define a concrete task:

> **Given the tokens that came before, predict the next token.**

This is called **next-token prediction**.

Mathematically, the model is trying to learn a conditional probability:

$$
P(x_{t+1} \mid x_1, x_2, ..., x_t)
$$

In words:

> What is the probability of the next token, given all the previous tokens?

This is the fundamental training objective behind autoregressive language models.

GPT models use this same basic objective. The major difference is that GPT uses a much more powerful architecture to calculate the prediction.

---

## What does `block_size` mean?

`block_size` represents the maximum amount of context that we are using in one training example.

Here:

```python
block_size = 8
```

so the model can work with sequences of up to 8 tokens.

It is useful to think of `block_size` as the **context length** of this small experiment.

For example:

```text
"The king"
```

could be represented by several character tokens, and the model can use the previous characters as context when trying to predict the next one.

---

## The key idea

At this point, we have transformed our raw text into a supervised learning problem:

```text
Raw text
   ↓
Token IDs
   ↓
Input / target pairs
   ↓
Predict the next token
```

This transformation is essential.

The model does not learn directly from the text as a string. It learns from many examples of the form:

```text
context → correct next token
```

Everything we do from this point onward is about making this prediction task more efficient and building a neural network that can solve it.


In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"when the context is {context} the target is {target}")

when the context is tensor([18]) the target is 47
when the context is tensor([18, 47]) the target is 56
when the context is tensor([18, 47, 56]) the target is 57
when the context is tensor([18, 47, 56, 57]) the target is 58
when the context is tensor([18, 47, 56, 57, 58]) the target is 1
when the context is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when the context is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when the context is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


# 9. Creating Batches of Training Examples

We now have individual input/target sequences.

However, during neural network training, we usually don't process only one sequence at a time.

Instead, we process several sequences simultaneously in a **batch**.

We define:

```python
batch_size = 4
block_size = 8
```

This means that each batch contains:

```text
4 sequences
×
8 token positions
```

So the input tensor will have shape:

```text
(B, T)
```

where:

* `B` = batch size
* `T` = number of time positions

In our case:

```text
B = 4
T = 8
```

so:

```text
xb.shape = (4, 8)
yb.shape = (4, 8)
```

---

## What does `get_batch()` do?

The function:

```python
get_batch(split)
```

randomly chooses several starting positions in either the training or validation data.

For each starting position, it extracts:

```text
block_size
```

tokens for the input.

Then it extracts the same sequence shifted by one position for the target.

Conceptually, for one sequence:

```text
x: [a, b, c, d, e, f, g, h]
y: [b, c, d, e, f, g, h, i]
```

The code:

```python
x = torch.stack([
    data[i:i+block_size]
    for i in ix
])
```

creates the input batch.

And:

```python
y = torch.stack([
    data[i+1:i+block_size+1]
    for i in ix
])
```

creates the corresponding target batch.

`torch.stack()` combines the individual sequences into a single tensor.

---

## Visualizing the batch

We can imagine `xb` as:

```text
                 time →
             1  2  3  4  5  6  7  8

sequence 1   •  •  •  •  •  •  •  •
sequence 2   •  •  •  •  •  •  •  •
sequence 3   •  •  •  •  •  •  •  •
sequence 4   •  •  •  •  •  •  •  •
```

Therefore:

```text
xb.shape = (4, 8)
```

and the target tensor `yb` has exactly the same shape:

```text
yb.shape = (4, 8)
```

But the values in `yb` are shifted by one token.

---

## Why use batches?

There are two major reasons.

### 1. Computational efficiency

Modern hardware, especially GPUs, is designed to perform many operations simultaneously.

Instead of processing:

```text
sequence 1
```

then:

```text
sequence 2
```

then:

```text
sequence 3
```

we process all of them together.

### 2. More training examples per update

Our batch contains:

$$
B \times T = 4 \times 8 = 32
$$

prediction positions.

So one forward pass gives the model **32 next-token prediction problems**.

This allows the loss to be calculated over many examples at once.

---

## Understanding the printed examples

The notebook then loops through:

```python
for b in range(batch_size):
    for t in range(block_size):
```

Here:

* `b` selects which sequence in the batch we are looking at.
* `t` selects which position within that sequence we are looking at.

Then:

```python
context = xb[b, :t+1]
target = yb[b, t]
```

means:

> Take all the tokens up to position `t` as the context, and use the corresponding token from `yb` as the correct answer.

For example:

```text
context = [57]
target = 1
```

Then:

```text
context = [57, 1]
target = 46
```

Then:

```text
context = [57, 1, 46]
target = 47
```

and so on.

This demonstrates something very important:

**One sequence creates multiple training examples.**

---

## The key idea

The batch gives us many examples of:

```text
context → next token
```

at the same time.

We now have everything necessary to feed data into a language model:

```text
text
 ↓
token IDs
 ↓
input/target pairs
 ↓
batches
 ↓
language model
```

The next cell simply displays `xb`, the actual input tensor that will be fed into our model.


In [ ]:
torch.manual_seed(42)
batch_size = 4
block_size = 8

def get_batch(split):
  # generate a small batch of data of inputs x and targets y
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,)) # this will generate a (4, 1) which is basically our (batch_size, 1) random number into lists
  # the number of ix is the starting point of the tokens we want to take from the sentence and then
  # we read the data and make our blocks
  x = torch.stack([data[i:i+block_size] for i in ix]) # to stack does vectors as ROWs
  y = torch.stack([data[i+1 : i+block_size+1] for i in ix]) # to stack does vectors as ROWs
  return x, y

xb, yb = get_batch('train')
print('inputs')
print(xb.shape)
print("--" * 40)
print({xb})
print('targets')
print(yb.shape)
print("--" * 40)
print(yb)

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b, :t+1]
    target = yb[b, t]
    print(f"when we have {context} we want to predict {target}")

inputs
torch.Size([4, 8])
--------------------------------------------------------------------------------
{tensor([[57,  1, 46, 47, 57,  1, 50, 53],
        [ 1, 58, 46, 43, 56, 43,  1, 41],
        [17, 26, 15, 17, 10,  0, 32, 53],
        [57, 58,  6,  1, 61, 47, 58, 46]])}
targets
torch.Size([4, 8])
--------------------------------------------------------------------------------
tensor([[ 1, 46, 47, 57,  1, 50, 53, 60],
        [58, 46, 43, 56, 43,  1, 41, 39],
        [26, 15, 17, 10,  0, 32, 53,  1],
        [58,  6,  1, 61, 47, 58, 46,  0]])
when we have tensor([57]) we want to predict 1
when we have tensor([57,  1]) we want to predict 46
when we have tensor([57,  1, 46]) we want to predict 47
when we have tensor([57,  1, 46, 47]) we want to predict 57
when we have tensor([57,  1, 46, 47, 57]) we want to predict 1
when we have tensor([57,  1, 46, 47, 57,  1]) we want to predict 50
when we have tensor([57,  1, 46, 47, 57,  1, 50]) we want to predict 53
when we have tensor([57,  1

# 10. The Input to the Language Model

The variable `xb` contains the input batch that we will feed into our language model.

Its shape is:

```text
(4, 8)
```

This means:

```text
4 sequences
×
8 token positions
```

For example, `xb` might look like:

```text
tensor([
    [57,  1, 46, 47, 57,  1, 50, 53],
    [ 1, 58, 46, 43, 56, 43,  1, 41],
    [17, 26, 15, 17, 10,  0, 32, 53],
    [57, 58,  6,  1, 61, 47, 58, 46]
])
```

These numbers are **token IDs**, not meaningful numerical quantities.

For example, from our `stoi` mapping:

```text
57 → 's'
1  → ' '
46 → 'h'
47 → 'i'
```

So a sequence such as:

```text
[57, 1, 46, 47]
```

represents:

```text
"s hi"
```

The important point is that the neural network receives the numerical IDs.

It does not receive the characters directly.

---

## What happens next?

The next cell creates our first language model:

```text
BigramLanguageModel
```

At this point, we are **not yet building a Transformer**.

The model is deliberately extremely simple.

It will allow us to understand:

1. how token IDs enter a neural network,
2. how predictions are produced,
3. what logits are,
4. how the loss is calculated,
5. and eventually how gradients can be used to train the model.

Once these ideas are clear, we can replace this simple model with a Transformer architecture.

---

## A useful mental model

Think of the current pipeline as:

```text
characters
    ↓
integer token IDs
    ↓
xb
    ↓
language model
    ↓
prediction for the next character
```

The next cell is where the actual neural network begins.


In [ ]:
# this is our input that we want to feed to the transformer
print(xb)

tensor([[57,  1, 46, 47, 57,  1, 50, 53],
        [ 1, 58, 46, 43, 56, 43,  1, 41],
        [17, 26, 15, 17, 10,  0, 32, 53],
        [57, 58,  6,  1, 61, 47, 58, 46]])


# 11. Building the Bigram Language Model

We now build our first actual language model.

This is a **Bigram Language Model**.

The name "bigram" comes from the fact that the model learns relationships between two consecutive tokens:

```text
current token → next token
```

For example:

```text
'h' → ?
```

The model needs to answer:

> Given that the current character is `'h'`, what character should come next?

Because our vocabulary contains 65 characters, the model needs to produce 65 possible scores.

---

# 11.1 The Model Architecture

The main component of the model is:

```text
nn.Embedding(vocab_size, vocab_size)
```

Since:

```text
vocab_size = 65
```

the embedding table has shape:

```text
(65, 65)
```

We can visualize this as a table:

```text
                         Possible next token
                    0    1    2    3   ...   64

Current token  0   [ ]  [ ]  [ ]  [ ]  ...  [ ]
Current token  1   [ ]  [ ]  [ ]  [ ]  ...  [ ]
Current token  2   [ ]  [ ]  [ ]  [ ]  ...  [ ]
     ...
Current token 64   [ ]  [ ]  [ ]  [ ]  ...  [ ]
```

Each row represents the current token.

Each column represents one possible next token.

Therefore, each row contains 65 numbers.

These numbers will become the model's **logits** for the next token.

---

# 11.2 Why Does `nn.Embedding` Work Here?

An embedding layer normally maps token IDs to learned vectors.

For example:

```text
token ID
   ↓
embedding vector
```

In a normal Transformer, these vectors might have dimensions such as:

```text
65 → 128 dimensions
```

or:

```text
50,000 → 768 dimensions
```

But here we deliberately use:

```text
65 → 65 dimensions
```

This makes the embedding table directly usable as a table of next-token scores.

So for a particular input token, the model simply looks up one row.

For example:

```text
current token = 46
```

The embedding layer retrieves:

```text
row 46
```

That row contains 65 numbers:

```text
[
    score for token 0,
    score for token 1,
    score for token 2,
    ...
    score for token 64
]
```

Those 65 values are the logits for the next-token prediction.

---

# 11.3 The Forward Pass

The `forward()` method receives two tensors:

```text
idx
targets
```

Both initially have shape:

```text
(B, T)
```

where:

* `B` = batch size
* `T` = sequence length

In our notebook:

```text
B = 4
T = 8
```

so:

```text
idx.shape = (4, 8)
targets.shape = (4, 8)
```

The first important operation is:

```text
logits = self.token_embedding_table(idx)
```

The embedding lookup transforms:

```text
(B, T)
```

into:

```text
(B, T, C)
```

where:

```text
C = vocab_size = 65
```

Therefore:

```text
(4, 8)
   ↓
(4, 8, 65)
```

This is a very important shape transformation.

It means:

```text
4 sequences
×
8 positions
×
65 possible next tokens
```

So the model is making:

$$
4 \times 8 = 32
$$

separate next-token predictions.

Each prediction has 65 possible classes.

---

# 11.4 What Are Logits?

The output of the model is called `logits`.

Logits are **raw prediction scores**.

They are not probabilities yet.

For example, suppose the model sees the character:

```text
'h'
```

It might produce:

```text
'a' → -1.2
'b' → -3.5
'c' → -2.1
'd' →  0.4
'e' →  4.7
...
```

The larger the logit, the more strongly the model favors that token.

At the beginning of training, these numbers are essentially random because the parameters of the model have not learned the statistics of Shakespeare yet.

After training, the model should assign higher scores to characters that are more likely to follow the current character.

---

# 11.5 Reshaping the Logits

Our logits currently have shape:

```text
(B, T, C)
```

or:

```text
(4, 8, 65)
```

However, `F.cross_entropy()` expects the class dimension to be the second dimension.

So we reshape the tensor:

```text
(4, 8, 65)
       ↓
(32, 65)
```

Why 32?

Because:

$$
4 \times 8 = 32
$$

There are 32 separate prediction problems.

Therefore:

```text
32 rows
×
65 possible classes
```

Each row corresponds to one prediction.

---

# 11.6 Reshaping the Targets

The target tensor initially has shape:

```text
(4, 8)
```

We reshape it into:

```text
(32)
```

Now the relationship between predictions and targets is:

```text
logits:

(32, 65)

targets:

(32)
```

For every one of the 32 predictions, we have exactly one correct token ID.

Conceptually:

```text
prediction 1 → correct token
prediction 2 → correct token
prediction 3 → correct token
...
prediction 32 → correct token
```

---

# 11.7 Cross Entropy Loss

We calculate:

```text
loss = F.cross_entropy(logits, targets)
```

The loss tells us how well the model predicted the correct next token.

Suppose the correct answer is token `43`.

If the model gives token `43` a high probability:

```text
token 43 → 0.80
```

then the model receives a relatively small loss.

If the model gives the correct token a very low probability:

```text
token 43 → 0.001
```

then the loss is much larger.

Therefore:

> **Lower loss means the model is assigning more probability to the correct answers.**

During training, we will use this loss to calculate gradients and update the model's parameters.

---

# 11.8 Logits Are Not Probabilities

This distinction is extremely important.

The model produces:

```text
logits
```

not:

```text
probabilities
```

For example:

```text
[2.1, 0.4, -1.2, 3.7]
```

These values do not sum to 1.

To convert them into probabilities, we use the **softmax** function:

```text
F.softmax(logits, dim=-1)
```

which might produce something like:

```text
[0.10, 0.02, 0.01, 0.87]
```

Now the values form a probability distribution and sum to 1.

The model's generation code uses softmax because it needs probabilities in order to sample a next token.

For the loss, however, we pass the raw logits directly to:

```text
F.cross_entropy()
```

because PyTorch's cross-entropy implementation handles the necessary normalization internally.

---

# 11.9 Why Is It Called a Bigram Model?

The model only learns local relationships:

```text
current token → next token
```

It does not yet have a mechanism for understanding long-range context.

For example, if the text contains:

```text
"The king went ..."
```

the model does not build a sophisticated representation of the entire phrase.

It mainly learns relationships between consecutive characters.

This is why it is called a **Bigram Language Model**.

It is extremely limited compared with GPT.

But it is useful because it isolates the fundamental language-modeling idea:

> Given some input, produce a probability distribution over the next token.

---

# 11.10 The Complete Forward Pass

The entire forward pass can be summarized as:

```text
Input token IDs
       │
       ▼
Embedding lookup
       │
       ▼
Logits
       │
       ▼
Reshape
       │
       ▼
Cross Entropy
       │
       ▼
Loss
```

With our actual tensor shapes:

```text
xb
(4, 8)
  │
  ▼
Embedding
  │
  ▼
logits
(4, 8, 65)
  │
  ▼
reshape
(32, 65)
  │
  ▼
cross entropy
  │
  ▼
loss
()
```

The final loss is a scalar.

In the current notebook, it is approximately:

```text
4.8865
```

The fact that the loss is relatively high at this point is expected because the model has not been trained yet.

---

# 11.11 The `generate()` Function

The model also contains a `generate()` method.

Its purpose is different from `forward()`.

### `forward()`

The forward pass answers:

> Given the correct input and target sequences, how well is the model performing?

It is primarily used during training and evaluation.

### `generate()`

Generation answers:

> Given some starting tokens, what token should the model generate next?

The general generation process is:

```text
starting context
       ↓
model predicts logits
       ↓
take the last time step
       ↓
convert logits to probabilities
       ↓
sample a token
       ↓
append the token to the sequence
       ↓
repeat
```

The line:

```text
logits = logits[:, -1, :]
```

selects only the prediction at the **last time position**.

Why?

Because when generating text, we only need the prediction for the next token after the current sequence.

Then:

```text
F.softmax(logits, dim=-1)
```

converts the logits into probabilities.

Finally:

```text
torch.multinomial(probs, num_samples=1)
```

samples one token according to those probabilities.

This makes generation stochastic rather than simply choosing the token with the largest probability every time.

---

# 11.12 Important Issues in the Current `generate()` Code

There are two things worth noticing in the current notebook.

First, the code calls:

```text
logits, loss = self(idx)
```

but the `forward()` method is defined as:

```text
forward(self, idx, targets)
```

Therefore, the current `generate()` implementation does not provide the required `targets` argument.

Second, the line:

```text
return idx
```

is currently inside the `for` loop.

That means the function returns after the **first iteration** instead of generating all `max_new_tokens`.

These are implementation issues in the current notebook rather than concepts of the Bigram model itself.

We can fix them later once we fully understand how `forward()` and `generate()` are supposed to work.

---

# 11.13 What Have We Built?

At this point, we have a complete but extremely simple autoregressive language model.

The complete pipeline is:

```text
Shakespeare text
       ↓
Characters
       ↓
Integer token IDs
       ↓
Input / target pairs
       ↓
Batches
       ↓
Bigram Language Model
       ↓
Logits
       ↓
Probabilities
       ↓
Next-token prediction
```

The fundamental learning objective is already the same basic objective used by GPT:

> **Predict the next token given the previous tokens.**

What changes when we move from this Bigram model to a GPT-style Transformer is **how the model calculates that prediction**.

The Bigram model uses a simple lookup table.

A Transformer uses:

* token embeddings
* positional information
* self-attention
* query, key, and value projections
* feed-forward networks
* residual connections
* layer normalization
* multiple Transformer blocks

So this tiny Bigram model should be viewed as our starting point.

It gives us the simplest possible environment in which we can understand:

1. what the input tokens are,
2. what targets are,
3. what logits are,
4. what probabilities are,
5. what the loss represents,
6. and how next-token prediction works.

Once these concepts are clear, we can progressively replace the simple lookup table with the components that eventually form a GPT architecture.


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)

class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    # each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, targets = None):
    # idx and targets are both (B,T)
    logits = self.token_embedding_table(idx) # (B,T,C) ==> (batch, time, channel) ==> (4, 8, 65)

    if targets is None:
      loss = None
    else:
      # for the cross entropy the channels should be our second dimentsion( ,C, )
      # so we will change the dimentions to (B*T, C) and make it a second dimentional thing
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      # the cross entropy is softmax + negative log-liklihood here
      loss = F.cross_entropy(logits, targets)
    return logits, loss        # 65 is our vocab_size

  # Generate new tokens one at a time from the current context.
  # `idx` is the starting context (the token IDs we give the model to begin generating from).
  # `max_new_tokens` tells the model how many new tokens to generate.
  # Each newly generated token is added to `idx` and then used as context to predict the next token.
  # After generation, the resulting token IDs can be decoded back into readable text.
  def generate(self, idx, max_new_tokens):
    # idx is (B,T) array of indices in the current context
    for _ in range(max_new_tokens):
      # get the predictions
      logits, loss = self(idx, targets=None) # Explicitly pass targets=None
      # focus only on the last time step
      logits = logits[:, -1, :] # becomes (B, C)
      # apply softmax to get the probability
      probs = F.softmax(logits, dim = -1)
      # sample from the distribution
      idx_next = torch.multinomial(probs, num_samples = 1)
      # append sampled index to the running sequence
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    return idx

## 12. Generating Text with the Bigram Language Model

We have now created our `BigramLanguageModel`. The next step is to actually use it to generate some text.

The code below creates a new model, gives it a starting token, asks it to generate 100 new tokens, and then converts those token IDs back into characters.

The important thing to understand is that **this model has not been trained yet**.

---

### The code

```python
m = BigramLanguageModel(vocab_size)
idx = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(idx=idx, max_new_tokens=100)[0].tolist()))

## 13. Training the Bigram Language Model

So far, our Bigram Language Model has **randomly initialized parameters**.

That is why the generated text looked like random characters:

```text
o$,q&IWqW&xtCjaB?ij&bYRGkF?b; f ,CbwhtERCIfuWr...

In [ ]:
m = BigramLanguageModel(vocab_size)
idx = torch.zeros((1,1), dtype = torch.long)
print(decode(m.generate(idx=idx, max_new_tokens=100)[0].tolist()))


o$,q&IWqW&xtCjaB?ij&bYRGkF?b; f ,CbwhtERCIfuWr,DzJERjhLlVaF&EjffPHDFcNoGIG'&$qXisWTkJPw
 ,b Xgx?D3sj


In [ ]:
#now lets train the model
# creating a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32
for i in range(1000):

  # sample a batch of data
  # by using get_batch we are randomly selecting parts of the text data as xb and yb
  # that why each time we do the training the loss can vary differently!
  xb, yb = get_batch("train")

  # evaluate the loss
  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
  if i % 1000 == 0: print(i)

print(loss.item())

0
3.764620304107666


## 14. Generating Text After Training

Now that we have trained the Bigram Language Model for 1,000 iterations, we can generate text again and compare it with the output we saw before training.

We use the same generation code:

```python
idx = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(idx=idx, max_new_tokens=100)[0].tolist()))

In [ ]:
idx = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(idx=idx, max_new_tokens=100)[0].tolist()))


Eu;:G,ul.OPA:Hu lwPczaqmKpc'w$nISlsnaq!xt: ghSLq'dqaF:m;hSOZNn'rg$ks3IMJ.JWdLRLJqG'cULgryQHbNLu'V,gr


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 32
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('tiny-shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed) # n_embed is number of embeding dimensions
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device = device))
        x = tok_emb + pos_emb
        logits = self.lm_head(x) # (B,T,vocab_size)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens to prevent position embedding out of bounds
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, : ] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 0: train loss 4.4801, val loss 4.4801
step 300: train loss 2.8827, val loss 2.9059
step 600: train loss 2.6890, val loss 2.7210
step 900: train loss 2.6000, val loss 2.6195
step 1200: train loss 2.5720, val loss 2.5858
step 1500: train loss 2.5365, val loss 2.5563
step 1800: train loss 2.5250, val loss 2.5383
step 2100: train loss 2.5161, val loss 2.5211
step 2400: train loss 2.5057, val loss 2.5183
step 2700: train loss 2.5127, val loss 2.5154
step 3000: train loss 2.4987, val loss 2.5132
step 3300: train loss 2.5009, val loss 2.5019
step 3600: train loss 2.4848, val loss 2.5093
step 3900: train loss 2.4973, val loss 2.5115
step 4200: train loss 2.4889, val loss 2.5071
step 4500: train loss 2.4800, val loss 2.4902
step 4800: train loss 2.4776, val loss 2.4978



CEThik bridcowindakis by ble

Hisenk Bbe d e.
S:
O:
ISM:


LUCous:
Wanthar u qur, vet?
F dilasoate awice my.
Whandacom oroup
Yowhthatof is h ble mil ndill, aes ireeesengcin lat Heriliov ts, and Win nghirileransesel lind m

In [ ]:
 # The mathematical trick in self-attention
# for example:
torch.manual_seed(42)
B,T,C = 4, 8, 2
x = torch.randn(B,T,C)
print(x)
x.shape

tensor([[[ 1.9269,  1.4873],
         [ 0.9007, -2.1055],
         [ 0.6784, -1.2345],
         [-0.0431, -1.6047],
         [-0.7521,  1.6487],
         [-0.3925, -1.4036],
         [-0.7279, -0.5594],
         [-0.7688,  0.7624]],

        [[ 1.6423, -0.1596],
         [-0.4974,  0.4396],
         [-0.7581,  1.0783],
         [ 0.8008,  1.6806],
         [ 1.2791,  1.2964],
         [ 0.6105,  1.3347],
         [-0.2316,  0.0418],
         [-0.2516,  0.8599]],

        [[-1.3847, -0.8712],
         [-0.2234,  1.7174],
         [ 0.3189, -0.4245],
         [ 0.3057, -0.7746],
         [-1.5576,  0.9956],
         [-0.8798, -0.6011],
         [-1.2742,  2.1228],
         [-1.2347, -0.4879]],

        [[-0.9138, -0.6581],
         [ 0.0780,  0.5258],
         [-0.4880,  1.1914],
         [-0.8140, -0.7360],
         [-1.4032,  0.0360],
         [-0.0635,  0.6756],
         [-0.0978,  1.8446],
         [-1.1845,  1.3835]]])


torch.Size([4, 8, 2])

In [ ]:
xbow = torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    xprev = x[b, :t+1]
    xbow[b,t] = torch.mean(xprev, 0)

```text```
xbow[0]:

position 0 = mean(token0)

position 1 = mean(token0, token1)

position 2 = mean(token0, token1, token2)

position 3 = mean(token0, token1, token2, token3)

...

position 7 = mean(token0,...,token7)





In [ ]:
xbow[0] # compare it with the x
# what is going on isthe the each ROW is on token and that two columns
# of that row is the features of that token
# this means that each token has two feature
# and eace token's features' values come from the mean of the previous tokens

tensor([[ 1.9269,  1.4873],
        [ 1.4138, -0.3091],
        [ 1.1687, -0.6176],
        [ 0.8657, -0.8644],
        [ 0.5422, -0.3617],
        [ 0.3864, -0.5354],
        [ 0.2272, -0.5388],
        [ 0.1027, -0.3762]])

In [ ]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('---------')
print('b=')
print(b)
print('---------')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
---------
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
---------
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [ ]:
weights = torch.tril(torch.ones(T,T))
# (B, T, C) @ (B, T, T) ---> (B, T, C)
weights = weights / weights.sum(1, keepdim=True)
xbow2 = weights @ x

In [ ]:
torch.allclose(xbow, xbow2)

True

In [ ]:
# wei === weights
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [ ]:
wei = F.softmax(wei, dim= 1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [ ]:
xbow3 = wei @ x

In [ ]:
torch.allclose(xbow, xbow3)

True

In [ ]:
# now we are adding the self-attention
torch.manual_seed(42)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C) # create a random number in this shape

# we make a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False) # W_k
query = nn.Linear(C, head_size, bias=False) # W_q
value = nn.Linear(C, head_size, bias=False) # W_v
k = key(x) # (B, T, 16) this also makes the
q = query(x) # (B, T, 16)
wei = q @ k.transpose(-2, -1) * head_size ** -0.5# (B, T, 16) @ (B, 16, T) ---> (B, T, T) this is (4, 8, 8)
# by wei, we created a token to token (one from the key and one from the query, comparison )
# -2 means we want the last two dimension


tril = torch.tril(torch.ones(T,T))

wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim = -1)

v = value(x)
out = wei @ v

print(wei.shape)

torch.Size([4, 8, 8])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 384
n_layer = 6 # number of blocks for the transformer
head_size = 6 # number of heads for the multi-head attention
dropout = 0.2
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('tiny-shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


# implement it into neural network

class Head(nn.Module):
  def __init__(self, head_size):
     super().__init__()
     self.key = nn.Linear(n_embed, head_size, bias=False)
     self.query = nn.Linear(n_embed, head_size, bias=False)
     self.value = nn.Linear(n_embed, head_size, bias=False)
     self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
     self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B, T, C = x.shape
    k = self.key(x) # (B, T, head_size)
    q = self.query(x) # (B, T, head_size)
    # compute attention scores ("affinities")
    wei = q @ k.transpose(-2, -1) * C ** -0.5 # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)
    # perform the weighted aggregation of the values
    v = self.value(x)
    out = wei @ v
    return out

class FeedForward(nn.Module):
  def __init__(self, n_embed):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embed, 4 * n_embed),
        nn.ReLU(),
        nn.Linear(4 * n_embed, n_embed),
        nn.Dropout(dropout)
    )
  def forward(self, x):
    return self.net(x)

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embed, n_embed)
    self.dropout = nn.Dropout(dropout)
  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.proj(out)
    out = self.dropout(out)
    return out

In [ ]:
class Block(nn.Module):
  # this is our transformer block
  def __init__(self, n_embed, n_head):
    super().__init__()
    head_size = n_embed // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.fwa = FeedForward(n_embed)
    self.ln1 = nn.LayerNorm(n_embed)
    self.ln2 = nn.LayerNorm(n_embed)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.fwa(self.ln2(x))
    return x


class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed) # n_embed is number of embeding dimensions
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[Block(n_embed, n_head = n_layer) for _ in range(n_layer)])
        # self.sa_head = MultiHeadAttention(4, n_embed // 4)
        # self.ffwd = FeedForward(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device = device))
        x = tok_emb + pos_emb
        x = self.blocks(x) # apply one head of self-attention
        logits = self.lm_head(x) # (B,T,vocab_size)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens to prevent position embedding out of bounds
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, : ] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


model = BigramLanguageModel()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


step 0: train loss 4.4753, val loss 4.4709
step 500: train loss 2.0850, val loss 2.1490
step 1000: train loss 1.6627, val loss 1.8234
step 1500: train loss 1.4919, val loss 1.6842
step 2000: train loss 1.3853, val loss 1.6081
step 2500: train loss 1.3170, val loss 1.5641
step 3000: train loss 1.2612, val loss 1.5324
step 3500: train loss 1.2158, val loss 1.5094
step 4000: train loss 1.1767, val loss 1.4957
step 4500: train loss 1.1384, val loss 1.4843

But with prison: I will steed win thou,
merchance, Carists, and love him, boys,
I more have borne more: if thou, let's procious by them
Would have become allies the senaty
of him, and trumpet of her recomplexions
And chrief: come to desire the joy
In this enemy.

Both the vood thing-for Ire courterfeitiving:
Ance in then, highing are crifted this morning.
Well murder, 'tis the one rinquest?

HENRY BOLINGBROKE:
Why no! may not we turn have standed, I do say the
YORK:
Why chisbuts: though thou unt


### Saving and Loading the Model

To save the trained model's weights so you don't lose progress, you can use PyTorch's `torch.save()`. Below is the code to save the model and an example of how to load it back.

In [ ]:
# Save the model's trained weights (state dictionary)
torch.save(model.state_dict(), 'nano_model.pt')
print("Model saved successfully as 'nano_model.pt'!")

Model saved successfully as 'nano_model.pt'!


In [ ]:
# To load the model later:
# 1. Re-instantiate the model structure
loaded_model = BigramLanguageModel()

# 2. Load the state dictionary into the model
loaded_model.load_state_dict(torch.load('nano_model.pt', map_location=device))
loaded_model.to(device)
loaded_model.eval() # set to evaluation mode
print("Model loaded successfully and ready!")

Model loaded successfully and ready!


### Using the Loaded Model for Text Generation

You can prompt your loaded model with any starting text you want. The model will encode your prompt, generate new characters based on it, and decode the result back into text.

In [ ]:
# 1. Define your starting prompt
starting_prompt = "ROMEO:\n"

# 2. Encode the prompt and convert it to a tensor with a batch dimension (B=1)
context_tokens = encode(starting_prompt)
context_tensor = torch.tensor([context_tokens], dtype=torch.long, device=device)

# 3. Generate new tokens using your loaded model
generated_tensor = loaded_model.generate(context_tensor, max_new_tokens=500)

# 4. Decode and print the final output
generated_text = decode(generated_tensor[0].tolist())
print(generated_text)

ROMEO:
My lord, I three kinsman! suspicion!
Give me the fruitful write molenty.
O, how any son hours both truth I live
On the time, and give my person of a bowl
Of early may despite in. God forwards, how can you,
Like from her: her, is the Tower.

BALTHASAR:
Get thy hand, to.

FRIAR LAURENCE:
I thought to remorn; your king.

POMPEY:
'Tis great ears to get repposition; and hours he may
serve: to the town I mean to you hath force these
commongs of them and take the fool. I will give me what I can fellow 


### Downloading the Model to Your Computer

To download the saved model file `gpt_model.pt` directly to your local computer from Google Colab, run the following cell:

In [ ]:
from google.colab import files

# Download the file to your local device
files.download('nano_model.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>